# TFM · Fase 4: motor de optimización y línea base greedy
Lee los backlogs normalizados de la Fase 3 (`salidas/E*/backlog.json`) y genera un plan con dos métodos:
- **M2 – greedy:** ordena por (v + λr)/p y asigna cada historia al primer sprint con capacidad, después de sus dependencias.
- **M3 – programación lineal entera:** ecuaciones (1)–(5) de la memoria, con PuLP y el solver CBC.

Calcula las métricas de la Tabla 4: brecha respecto al óptimo, valor entregado temprano, utilización, violaciones, historias sin asignar y tiempo de cómputo.

## 1. Instalación
Las versiones recientes de PuLP ya no incluyen CBC; el extra `[cbc]` lo instala.

In [11]:
##!pip install -q "pulp[cbc]"
%pip install nbformat plotly

Note: you may need to restart the kernel to use updated packages.


## 2. Módulo del motor

In [1]:
%%writefile motor_planificacion.py
"""Motor de planificación de sprints — TFM, Fase 4.

Asigna las historias del backlog a sprints con dos métodos:
  - M3: programación lineal entera (PuLP + CBC), ecuaciones (1)-(5) de la memoria.
  - M2: heurística greedy por razón valor ponderado / story points (línea base).
Calcula las métricas de calidad de la Tabla 4 para cualquier plan (también para el manual, M1).

Entrada: lista de historias con claves 'clave', 'story_points', 'valor_negocio',
'prioridad_num' y 'depende_de' (formato de salida de jira_ingesta, Fase 3).
"""
import math
import time

import pandas as pd
import pulp


# --------------------------------------------------------------------------- datos
def preparar(backlog):
    """Normaliza la entrada (DataFrame o lista de dicts) y descarta dependencias externas."""
    filas = backlog.to_dict("records") if isinstance(backlog, pd.DataFrame) else list(backlog)
    incompletas = [f["clave"] for f in filas
                   if pd.isna(f.get("story_points")) or pd.isna(f.get("valor_negocio"))
                   or float(f["story_points"]) <= 0]
    if incompletas:   # sin estimación o sin valor no se pueden planificar (la Fase 3 ya las reporta)
        print(f"Aviso: se excluyen {len(incompletas)} historias sin story points o sin valor: {incompletas}")
    filas = [f for f in filas if f["clave"] not in incompletas]
    claves = {f["clave"] for f in filas}
    historias = {}
    for f in filas:
        deps = f.get("depende_de") or []
        if isinstance(deps, str):
            deps = [d.strip() for d in deps.split(",") if d.strip()]
        historias[f["clave"]] = {
            "p": float(f["story_points"]),
            "v": float(f["valor_negocio"]),
            "r": float(f.get("prioridad_num") or 0),
            "deps": [d for d in deps if d in claves],
        }
    return historias


def profundidad(historias):
    """Longitud de la cadena de dependencias más larga (nº de sprints mínimo por dependencias)."""
    nivel = {}

    def calc(h):
        if h not in nivel:
            nivel[h] = 1 + max((calc(k) for k in historias[h]["deps"]), default=0)
        return nivel[h]

    return max((calc(h) for h in historias), default=0)


def horizonte(historias, capacidad, ajustar_dependencias=False):
    """Mínimo teórico de sprints + 1 de holgura (diseño experimental).

    Con ajustar_dependencias=True no baja de la profundidad del grafo de dependencias,
    porque una historia solo puede ir en un sprint posterior al de las historias de las que depende.
    """
    total = sum(h["p"] for h in historias.values())
    n = math.ceil(total / capacidad) + 1
    return max(n, profundidad(historias)) if ajustar_dependencias else n


def pesos_sprint(n):
    """w_s = (n - s + 1) / n: decrece con cada sprint."""
    return {s: (n - s + 1) / n for s in range(1, n + 1)}


def valor_ponderado(h, lam):
    return h["v"] + lam * h["r"]


# --------------------------------------------------------------------------- M3: PLE
def _binaria(prob, nombre):
    """Variable binaria compatible con PuLP 2.x (LpVariable) y 4.x (add_variable)."""
    if hasattr(prob, "add_variable"):
        return prob.add_variable(nombre, cat="Binary")
    return pulp.LpVariable(nombre, cat="Binary")


def _solver_cbc(mensajes, limite_s, warm=False):
    """CBC: PULP_CBC_CMD en PuLP 2.x; COIN_CMD (paquete pulp[cbc]) en PuLP 3+/4.x."""
    clase = getattr(pulp, "PULP_CBC_CMD", None) or pulp.COIN_CMD
    return clase(msg=mensajes, timeLimit=limite_s, warmStart=warm, gapRel=0)   # óptimo probado


def planificar_ple(historias, capacidad, n_sprints, lam=10.0, limite_s=300, mensajes=False, arranque=True):
    S = list(range(1, n_sprints + 1))
    w = pesos_sprint(n_sprints)
    H = list(historias)
    prob = pulp.LpProblem("planificacion_sprints", pulp.LpMaximize)
    x = {h: {s: _binaria(prob, f"x_{i}_{s}") for s in S} for i, h in enumerate(H)}        # (5)
    prob += pulp.lpSum(valor_ponderado(historias[h], lam) * w[s] * x[h][s]
                       for h in H for s in S)                                              # (1)
    for h in H:
        prob += pulp.lpSum(x[h][s] for s in S) <= 1, f"unica_{h}"                          # (2)
    for s in S:
        prob += pulp.lpSum(historias[h]["p"] * x[h][s] for h in H) <= capacidad, f"cap_{s}"  # (3)
    for h in H:
        for k in historias[h]["deps"]:
            for s in S:                                                                    # (4)
                prob += (pulp.lpSum(x[h][t] for t in S if t <= s)
                         <= pulp.lpSum(x[k][t] for t in S if t < s)), f"dep_{h}_{k}_{s}"
    t0 = time.perf_counter()
    if arranque:   # solución inicial factible: la del greedy (acelera la búsqueda de CBC)
        inicial, _ = planificar_greedy(historias, capacidad, n_sprints, lam)
        for h in H:
            for s in S:
                x[h][s].setInitialValue(1 if inicial[h] == s else 0)
    resultado = prob.solve(_solver_cbc(mensajes, limite_s, warm=arranque))
    segundos = time.perf_counter() - t0
    plan = {h: next((s for s in S if (x[h][s].value() or 0) > 0.5), None) for h in H}
    return plan, {"estado": _estado(prob, resultado), "segundos": segundos,
                  "variables": len(H) * len(S), "restricciones": prob.numConstraints() if hasattr(prob, "numConstraints") else len(prob.constraints)}


def _estado(prob, resultado):
    """Texto del estado de la solución en PuLP 2.x (LpStatus) y 4.x (LpSolveStats)."""
    if hasattr(pulp, "LpStatus"):
        return pulp.LpStatus[prob.status]
    nombre = getattr(getattr(resultado, "status", None), "name", str(resultado))
    return "Optimal" if nombre == "GapLimit" else nombre   # con gapRel=0, GapLimit = óptimo probado


# --------------------------------------------------------------------------- M2: greedy
def planificar_greedy(historias, capacidad, n_sprints, lam=10.0):
    t0 = time.perf_counter()
    orden = sorted(historias, key=lambda h: (-valor_ponderado(historias[h], lam) / historias[h]["p"], h))
    carga = {s: 0.0 for s in range(1, n_sprints + 1)}
    plan = {h: None for h in historias}
    cambio = True
    while cambio:                       # se repite hasta que no sea posible asignar más historias
        cambio = False
        for h in orden:
            if plan[h] is not None:
                continue
            deps = historias[h]["deps"]
            if any(plan[k] is None for k in deps):
                continue
            desde = max((plan[k] for k in deps), default=0) + 1
            for s in range(desde, n_sprints + 1):
                if carga[s] + historias[h]["p"] <= capacidad:
                    plan[h], carga[s], cambio = s, carga[s] + historias[h]["p"], True
                    break
    return plan, {"estado": "Heuristic", "segundos": time.perf_counter() - t0}


# --------------------------------------------------------------------------- métricas (Tabla 4)
def metricas(historias, plan, capacidad, n_sprints, lam=10.0):
    w = pesos_sprint(n_sprints)
    asignadas = {h: s for h, s in plan.items() if s is not None}
    z = sum(valor_ponderado(historias[h], lam) * w[s] for h, s in asignadas.items())
    carga = {s: 0.0 for s in range(1, n_sprints + 1)}
    for h, s in asignadas.items():
        carga[s] += historias[h]["p"]
    mitad = n_sprints // 2
    valor_total = sum(h["v"] for h in historias.values())
    viol_dep = sum(1 for h, s in asignadas.items() for k in historias[h]["deps"]
                   if plan.get(k) is None or plan[k] >= s)
    return {
        "Z": round(z, 2),
        "valor_temprano_pct": round(100 * sum(historias[h]["v"] for h, s in asignadas.items() if s <= mitad)
                                    / valor_total, 1),
        "utilizacion_pct": round(100 * sum(carga.values()) / (capacidad * n_sprints), 1),
        "sprints_sobrecargados": sum(1 for c in carga.values() if c > capacidad + 1e-9),
        "dependencias_violadas": viol_dep,
        "sin_asignar": len(historias) - len(asignadas),
        "sp_sin_asignar": sum(historias[h]["p"] for h in historias if plan.get(h) is None),
    }


def plan_a_tabla(historias, plan):
    return pd.DataFrame([{"clave": h, "sprint": s, "story_points": historias[h]["p"],
                          "valor_negocio": historias[h]["v"], "prioridad_num": historias[h]["r"],
                          "depende_de": ",".join(historias[h]["deps"])}
                         for h, s in plan.items()]).sort_values(["sprint", "clave"], na_position="last")

def plan_a_tabla_estructurada(backlog_original, plan):
    """
    Cruza el plan generado con el backlog original para exportar un CSV legible,
    incluyendo títulos, formato amigable de sprints y ordenado por prioridad.
    """
    # Convertir a lista de diccionarios si viene como DataFrame
    if isinstance(backlog_original, pd.DataFrame):
        filas = backlog_original.to_dict("records")
    else:
        filas = list(backlog_original)
        
    datos_csv = []
    
    for fila in filas:
        clave = fila["clave"]
        sprint_num = plan.get(clave)
        
        # Formatear la columna Sprint 
        nombre_sprint = f"Sprint {int(sprint_num)}" if sprint_num is not None else "Sin asignar"
        
        # Limpiar dependencias para el CSV
        deps = fila.get("depende_de", [])
        deps_str = ", ".join(deps) if isinstance(deps, list) else str(deps)
        
        datos_csv.append({
            "Sprint": nombre_sprint,
            "Clave": clave,
            "Resumen (Título)": fila.get("resumen", ""),
            "Story Points": fila.get("story_points", 0),
            "Valor de Negocio": fila.get("valor_negocio", 0),
            "Prioridad": fila.get("prioridad", ""),
            "Prioridad (Num)": fila.get("prioridad_num", 0),
            "Depende de": deps_str
        })
        
    df_export = pd.DataFrame(datos_csv)
    
    # Crear una columna temporal para ordenar correctamente: Sprints 1..N y luego "Sin asignar"
    df_export["_sprint_orden"] = df_export["Sprint"].str.extract(r'(\d+)').astype(float)
    
    # Ordenar por Sprint (ascendente) y luego por Valor de Negocio (descendente) para ver lo más valioso arriba
    df_export = df_export.sort_values(by=["_sprint_orden", "Valor de Negocio"], ascending=[True, False])
    
    # Limpiar columna temporal
    df_export = df_export.drop(columns=["_sprint_orden"])
    
    return df_export


def carga_por_sprint(historias, plan, n_sprints):
    carga = {s: 0.0 for s in range(1, n_sprints + 1)}
    for h, s in plan.items():
        if s is not None:
            carga[s] += historias[h]["p"]
    return carga


# --------------------------------------------------------------------------- comparación
def comparar(backlog, capacidad=25, lam=10.0, repeticiones=5, limite_s=300, n_sprints=None,
             ajustar_dependencias=False):
    """Ejecuta M2 y M3 'repeticiones' veces y devuelve métricas, planes e info de cómputo."""
    historias = preparar(backlog)
    n = n_sprints or horizonte(historias, capacidad, ajustar_dependencias)
    filas, planes = [], {}
    for metodo, fn in (("M2 greedy", planificar_greedy), ("M3 PLE", planificar_ple)):
        tiempos = []
        for _ in range(repeticiones):
            kw = {"limite_s": limite_s} if metodo == "M3 PLE" else {}
            plan, info = fn(historias, capacidad, n, lam, **kw)
            tiempos.append(info["segundos"])
        planes[metodo] = plan
        filas.append({"metodo": metodo, "estado": info["estado"], "sprints": n,
                      "tiempo_medio_s": round(sum(tiempos) / len(tiempos), 4),
                      **metricas(historias, plan, capacidad, n, lam)})
    tabla = pd.DataFrame(filas)
    z_opt = tabla.loc[tabla.metodo == "M3 PLE", "Z"].iloc[0]
    tabla["pct_del_optimo"] = (100 * tabla["Z"] / z_opt).round(2)
    tabla["brecha_pct"] = (100 - tabla["pct_del_optimo"]).round(2)
    return tabla, planes, historias, n


def curva_valor(historias, plan, n_sprints):
    """Porcentaje acumulado del valor de negocio total entregado al final de cada sprint."""
    total = sum(h["v"] for h in historias.values())
    acum, curva = 0.0, {}
    for s in range(1, n_sprints + 1):
        acum += sum(historias[h]["v"] for h, sp in plan.items() if sp == s)
        curva[s] = round(100 * acum / total, 1)
    return curva



Overwriting motor_planificacion.py


## 3. Datos de entrada
Sube el zip de la Fase 3 (`fase3_extraccion.zip`) con el icono de carpeta, o monta Drive. La celda lo descomprime en `salidas/`.

In [2]:
import importlib, json, os, zipfile
import pandas as pd
import motor_planificacion as mp
importlib.reload(mp)

if os.path.exists("fase3_extraccion.zip") and not os.path.exists("salidas/E1"):
    zipfile.ZipFile("fase3_extraccion.zip").extractall("salidas")
# Alternativa con Drive:
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.copytree("/content/drive/MyDrive/TFM/fase3", "salidas", dirs_exist_ok=True)

ESCENARIOS   = ["E1", "E2", "E3"]
VELOCIDAD    = 25      # capacidad C por sprint
LAMBDA       = 10      # peso de la prioridad frente al valor de negocio
LIMITE_S     = 300     # tiempo máximo de CBC por ejecución (segundos)
REPETICIONES = 5       # repeticiones para el tiempo medio (diseño experimental)
AJUSTAR_HORIZONTE = False   # True: horizonte >= profundidad del grafo de dependencias

BACKLOGS = {}
for e in ESCENARIOS:
    with open(f"salidas/{e}/backlog.json", encoding="utf-8") as f:
        BACKLOGS[e] = json.load(f)
    print(f"{e}: {len(BACKLOGS[e])} historias")

E1: 15 historias
E2: 30 historias
E3: 80 historias


## 4. Horizonte y profundidad de dependencias
Si la cadena de dependencias más larga supera el horizonte, algunas historias no caben con ningún método.

In [3]:
filas = []
for e, bl in BACKLOGS.items():
    H = mp.preparar(bl)
    filas.append({"escenario": e, "historias": len(H), "story_points": sum(h["p"] for h in H.values()),
                  "horizonte_diseño": mp.horizonte(H, VELOCIDAD),
                  "profundidad_dependencias": mp.profundidad(H),
                  "horizonte_ajustado": mp.horizonte(H, VELOCIDAD, ajustar_dependencias=True)})
pd.DataFrame(filas)

,escenario,historias,story_points,horizonte_diseño,profundidad_dependencias,horizonte_ajustado
0,E1,15,63.0,4,4,4
1,E2,30,139.0,7,9,9
2,E3,80,383.0,17,10,17


## 5. Comparar M2 y M3 en los tres escenarios
E3 puede tardar hasta `LIMITE_S` segundos por repetición. Si quieres una prueba rápida, baja `REPETICIONES` a 1.

In [4]:
RESULTADOS, PLANES = [], {}
for e, bl in BACKLOGS.items():
    rep = REPETICIONES if e != "E3" else 1
    tabla, planes, H, n = mp.comparar(bl, VELOCIDAD, LAMBDA, repeticiones=rep, limite_s=LIMITE_S,
                                      ajustar_dependencias=AJUSTAR_HORIZONTE)
    tabla.insert(0, "escenario", e)
    RESULTADOS.append(tabla)
    PLANES[e] = (planes, H, n)
    os.makedirs(f"planes/{e}", exist_ok=True)
    for m, pl in planes.items():
        mp.plan_a_tabla(H, pl).to_csv(f"planes/{e}/plan_{m.split()[0]}.csv", index=False, encoding="utf-8-sig")
        mp.plan_a_tabla_estructurada(bl, pl).to_csv(f"planes/{e}/plan_{m.split()[0]}.csv", index=False, encoding="utf-8-sig")
    print(f"{e} listo")
resultados = pd.concat(RESULTADOS)
resultados.to_csv("planes/resultados_motor.csv", index=False, encoding="utf-8-sig")
resultados  

c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\site-packages\pulp\apis\coin.py:130: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(


E1 listo


c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\site-packages\pulp\apis\coin.py:130: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(


E2 listo


c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\site-packages\pulp\apis\coin.py:130: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(


E3 listo


,escenario,metodo,estado,sprints,tiempo_medio_s,Z,valor_temprano_pct,utilizacion_pct,sprints_sobrecargados,dependencias_violadas,sin_asignar,sp_sin_asignar,pct_del_optimo,brecha_pct
0,E1,M2 greedy,Heuristic,4,0.0000,1116.25,84.7,63.0,0,0,0,0.0,100.00,0.00
1,E1,M3 PLE,Optimal,4,0.1028,1116.25,84.7,63.0,0,0,0,0.0,100.00,0.00
0,E2,M2 greedy,Heuristic,7,0.0001,1417.86,43.0,65.7,0,0,6,24.0,99.75,0.25
1,E2,M3 PLE,Optimal,7,0.1050,1421.43,43.9,65.7,0,0,6,24.0,100.00,0.00
0,E3,M2 greedy,Heuristic,17,0.0002,4206.59,57.0,89.6,0,0,1,2.0,93.52,6.48
1,E3,M3 PLE,TimeLimit,17,300.7366,4497.88,71.9,90.1,0,0,0,0.0,100.00,0.00


## 6. Capacidad por sprint y valor acumulado

In [25]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots

ESCENARIO_VER = "E2"
planes, H, n = PLANES[ESCENARIO_VER]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Story points por sprint", "Valor acumulado (%)"))
for m, pl in planes.items():
    carga = mp.carga_por_sprint(H, pl, n)
    curva = mp.curva_valor(H, pl, n)
    fig.add_bar(x=list(carga), y=list(carga.values()), name=m, row=1, col=1)
    fig.add_scatter(x=list(curva), y=list(curva.values()), mode="lines+markers", name=m, row=1, col=2)
fig.add_hline(y=VELOCIDAD, line_dash="dash", annotation_text="Capacidad", row=1, col=1)
fig.update_layout(title=f"{ESCENARIO_VER}: M2 greedy vs M3 PLE", barmode="group", height=420)
fig.show()

## 7. Sensibilidad al peso λ

In [26]:
sens = []
for e in ["E1", "E2"]:
    for lam in (0, 10, 25):
        t, _, _, _ = mp.comparar(BACKLOGS[e], VELOCIDAD, lam, repeticiones=1, limite_s=LIMITE_S)
        t.insert(0, "lambda", lam); t.insert(0, "escenario", e)
        sens.append(t)
sensibilidad = pd.concat(sens)[["escenario", "lambda", "metodo", "Z", "pct_del_optimo",
                                 "valor_temprano_pct", "sin_asignar"]]
sensibilidad.to_csv("planes/sensibilidad_lambda.csv", index=False, encoding="utf-8-sig")
sensibilidad

c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\site-packages\pulp\apis\coin.py:130: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(
c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\site-packages\pulp\apis\coin.py:130: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(
c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\site-packages\pulp\apis\coin.py:130: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(
c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\site-packages\pulp\apis\coin.py:130: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(
c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\site-packages\pulp\apis\coin.py:130: UserWarning: When using CBC on Windows, warmStart requires keepFiles=True.
  warnings.warn(
c:\Users\kamil\AppData\Local\Programs\Python\Python315\Lib\s

,escenario,lambda,metodo,Z,pct_del_optimo,valor_temprano_pct,sin_asignar
0,E1,0,M2 greedy,701.25,100.00,84.7,0
1,E1,0,M3 PLE,701.25,100.00,84.7,0
0,E1,10,M2 greedy,1116.25,100.00,84.7,0
1,E1,10,M3 PLE,1116.25,100.00,84.7,0
0,E1,25,M2 greedy,1738.75,100.00,84.7,0
1,E1,25,M3 PLE,1738.75,100.00,84.7,0
0,E2,0,M2 greedy,909.29,99.77,43.0,6
1,E2,0,M3 PLE,911.43,100.00,43.9,6
0,E2,10,M2 greedy,1417.86,99.75,43.0,6
1,E2,10,M3 PLE,1421.43,100.00,43.9,6


## 8. Descargar resultados

In [ ]:
import shutil
shutil.copy("motor_planificacion.py", "planes/motor_planificacion.py")
shutil.make_archive("fase4_motor", "zip", "planes")
#from google.colab import files
#files.download("fase4_motor.zip")

'd:\\Estudio\\UNIR\\TFM\\TFM-main\\TFM-main\\fase4_motor.zip'